In [ ]:
import pandas as pd

df = pd.read_excel("/content/Merged_Higher_Education_Dashboard_50.xlsx")

df.head()

,University,Country,QS_Rank,Overall_Score_QS,Academic_Reputation,Employer_Reputation,Citations_per_Faculty,Faculty_Student_Ratio,International_Students,THE_Rank,Overall_Score,Teaching,Research,Citations,Industry_Income,International_Outlook
0,MIT,USA,1,100.0,100,99,98,97,96,1,99.5,99,98,99,95,97
1,University of Oxford,UK,2,99.2,99,98,97,96,95,2,98.8,98,97,99,94,96
2,Stanford University,USA,3,98.4,99,98,97,96,95,4,98.0,98,97,98,94,96
3,Harvard University,USA,4,97.6,98,97,96,95,94,6,97.2,97,96,98,93,95
4,University of Cambridge,UK,5,96.8,98,97,96,95,94,3,96.5,97,96,98,93,95


In [ ]:
import pandas as pd

# Load dataset
df = pd.read_excel("/content/Merged_Higher_Education_Dashboard_50.xlsx")

# Check dataset
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

# Display first 5 rows
df.head()

Rows: 50
Columns: 16


,University,Country,QS_Rank,Overall_Score_QS,Academic_Reputation,Employer_Reputation,Citations_per_Faculty,Faculty_Student_Ratio,International_Students,THE_Rank,Overall_Score,Teaching,Research,Citations,Industry_Income,International_Outlook
0,MIT,USA,1,100.0,100,99,98,97,96,1,99.5,99,98,99,95,97
1,University of Oxford,UK,2,99.2,99,98,97,96,95,2,98.8,98,97,99,94,96
2,Stanford University,USA,3,98.4,99,98,97,96,95,4,98.0,98,97,98,94,96
3,Harvard University,USA,4,97.6,98,97,96,95,94,6,97.2,97,96,98,93,95
4,University of Cambridge,UK,5,96.8,98,97,96,95,94,3,96.5,97,96,98,93,95


In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

# =========================================================
# 1. LOAD YOUR DATASET
# =========================================================

input_file = "/content/Merged_Higher_Education_Dashboard_50.xlsx"

df = pd.read_excel(input_file)

print("Dataset loaded successfully")
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

print("\nColumns in dataset:")
print(df.columns.tolist())


# =========================================================
# 2. CLEAN COLUMN NAMES
# =========================================================

df.columns = (
    df.columns
    .str.strip()
    .str.replace("\n", " ", regex=False)
    .str.replace("  ", " ", regex=False)
)

print("\nCleaned columns:")
print(df.columns.tolist())


# =========================================================
# 3. FIND COLUMNS AUTOMATICALLY
# =========================================================

def find_column(names):

    for name in names:
        for col in df.columns:
            if col.lower().strip() == name.lower().strip():
                return col

    for name in names:
        for col in df.columns:
            if name.lower() in col.lower():
                return col

    return None


rank_col = find_column([
    "Rank",
    "World Rank",
    "World University Rank",
    "Overall Rank"
])

overall_col = find_column([
    "Overall Score",
    "Overall"
])

teaching_col = find_column([
    "Teaching",
    "Teaching Score"
])

research_col = find_column([
    "Research",
    "Research Score",
    "Research Environment"
])

citations_col = find_column([
    "Citations",
    "Citations Score"
])

international_col = find_column([
    "International Outlook",
    "International Outlook Score"
])

industry_col = find_column([
    "Industry Income",
    "Industry Income Score"
])

student_staff_col = find_column([
    "Student-Staff Ratio",
    "Student Staff Ratio",
    "Student-to-Staff Ratio"
])


print("\nDetected columns")
print("-------------------------")
print("Rank              :", rank_col)
print("Overall           :", overall_col)
print("Teaching          :", teaching_col)
print("Research          :", research_col)
print("Citations         :", citations_col)
print("International     :", international_col)
print("Industry          :", industry_col)
print("Student-Staff     :", student_staff_col)


# =========================================================
# 4. CONVERT NUMERIC COLUMNS
# =========================================================

columns_to_convert = [
    rank_col,
    overall_col,
    teaching_col,
    research_col,
    citations_col,
    international_col,
    industry_col,
    student_staff_col
]

for col in columns_to_convert:

    if col is not None:

        df[col] = (
            df[col]
            .astype(str)
            .str.replace("%", "", regex=False)
            .str.replace(",", "", regex=False)
            .str.strip()
        )

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )


# =========================================================
# 5. GLOBAL RANKING SCORE
# =========================================================

if rank_col is not None:

    min_rank = df[rank_col].min()
    max_rank = df[rank_col].max()

    if max_rank != min_rank:

        df["Global Ranking Score"] = (
            (max_rank - df[rank_col])
            /
            (max_rank - min_rank)
        ) * 100

    else:

        df["Global Ranking Score"] = 100


# =========================================================
# 6. RESEARCH IMPACT SCORE
# =========================================================

research_values = []

if research_col is not None:
    research_values.append(df[research_col])

if citations_col is not None:
    research_values.append(df[citations_col])

if research_values:

    df["Research Impact Score"] = pd.concat(
        research_values,
        axis=1
    ).mean(axis=1)


# =========================================================
# 7. TEACHING QUALITY SCORE
# =========================================================

if teaching_col is not None:

    df["Teaching Quality Score"] = df[teaching_col]


# =========================================================
# 8. INTERNATIONALIZATION SCORE
# =========================================================

if international_col is not None:

    df["Internationalization Score"] = (
        df[international_col]
    )


# =========================================================
# 9. INDUSTRY COLLABORATION SCORE
# =========================================================

if industry_col is not None:

    df["Industry Collaboration Score"] = (
        df[industry_col]
    )


# =========================================================
# 10. FACULTY-TO-STUDENT RATIO
# =========================================================

if student_staff_col is not None:

    df["Faculty-to-Student Ratio"] = (
        1 / df[student_staff_col]
    )


# =========================================================
# 11. NORMALIZATION FUNCTION
# =========================================================

def normalize_100(series):

    minimum = series.min()
    maximum = series.max()

    if maximum == minimum:
        return pd.Series(
            100,
            index=series.index
        )

    return (
        (series - minimum)
        /
        (maximum - minimum)
    ) * 100


# =========================================================
# 12. CREATE NORMALIZED KPI SCORES
# =========================================================

if "Global Ranking Score" in df.columns:

    df["Ranking KPI"] = normalize_100(
        df["Global Ranking Score"]
    )


if "Research Impact Score" in df.columns:

    df["Research KPI"] = normalize_100(
        df["Research Impact Score"]
    )


if "Teaching Quality Score" in df.columns:

    df["Teaching KPI"] = normalize_100(
        df["Teaching Quality Score"]
    )


if "Internationalization Score" in df.columns:

    df["International KPI"] = normalize_100(
        df["Internationalization Score"]
    )


if "Industry Collaboration Score" in df.columns:

    df["Industry KPI"] = normalize_100(
        df["Industry Collaboration Score"]
    )


# =========================================================
# 13. OVERALL KPI SCORE
# =========================================================

kpi_columns = [
    "Ranking KPI",
    "Research KPI",
    "Teaching KPI",
    "International KPI",
    "Industry KPI"
]

available_kpis = [
    col for col in kpi_columns
    if col in df.columns
]

df["Overall KPI Score"] = (
    df[available_kpis]
    .mean(axis=1)
)


# =========================================================
# 14. KPI RANK
# =========================================================

df["KPI Rank"] = (
    df["Overall KPI Score"]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)


# =========================================================
# 15. PERFORMANCE CATEGORY
# =========================================================

def performance_category(score):

    if score >= 80:
        return "Excellent"

    elif score >= 60:
        return "Good"

    elif score >= 40:
        return "Average"

    else:
        return "Needs Improvement"


df["Performance Category"] = (
    df["Overall KPI Score"]
    .apply(performance_category)
)


# =========================================================
# 16. SORT BY KPI RANK
# =========================================================

df = df.sort_values(
    "KPI Rank"
).reset_index(drop=True)


# =========================================================
# 17. ROUND KPI VALUES
# =========================================================

for col in [
    "Global Ranking Score",
    "Research Impact Score",
    "Faculty-to-Student Ratio",
    "Teaching Quality Score",
    "Internationalization Score",
    "Industry Collaboration Score",
    "Ranking KPI",
    "Research KPI",
    "Teaching KPI",
    "International KPI",
    "Industry KPI",
    "Overall KPI Score"
]:

    if col in df.columns:
        df[col] = df[col].round(2)


# =========================================================
# 18. DISPLAY FINAL KPI RESULTS
# =========================================================

print("\n======================================")
print("KPI ENGINEERING COMPLETED")
print("======================================")

print("Total universities:", len(df))

print("\nKPI columns created:")

for col in df.columns:

    if "KPI" in col or "Score" in col or "Ratio" in col:
        print("-", col)


# Display top 10
print("\nTop 10 Universities:")
display(df.head(10))


# =========================================================
# 19. SAVE KPI DATASET
# =========================================================

output_file = (
    "/content/Higher_Education_51_University_KPI.xlsx"
)

df.to_excel(
    output_file,
    index=False
)

print("\nFile created successfully:")
print(output_file)


# =========================================================
# 20. DOWNLOAD EXCEL FILE
# =========================================================

files.download(output_file)

Dataset loaded successfully
Number of rows: 50
Number of columns: 16

Columns in dataset:
['University', 'Country', 'QS_Rank', 'Overall_Score_QS', 'Academic_Reputation', 'Employer_Reputation', 'Citations_per_Faculty', 'Faculty_Student_Ratio', 'International_Students', 'THE_Rank', 'Overall_Score', 'Teaching', 'Research', 'Citations', 'Industry_Income', 'International_Outlook']

Cleaned columns:
['University', 'Country', 'QS_Rank', 'Overall_Score_QS', 'Academic_Reputation', 'Employer_Reputation', 'Citations_per_Faculty', 'Faculty_Student_Ratio', 'International_Students', 'THE_Rank', 'Overall_Score', 'Teaching', 'Research', 'Citations', 'Industry_Income', 'International_Outlook']

Detected columns
-------------------------
Rank              : QS_Rank
Overall           : Overall_Score_QS
Teaching          : Teaching
Research          : Research
Citations         : Citations
International     : None
Industry          : None
Student-Staff     : None

KPI ENGINEERING COMPLETED
Total universit

,University,Country,QS_Rank,Overall_Score_QS,Academic_Reputation,Employer_Reputation,Citations_per_Faculty,Faculty_Student_Ratio,International_Students,THE_Rank,...,International_Outlook,Global Ranking Score,Research Impact Score,Teaching Quality Score,Ranking KPI,Research KPI,Teaching KPI,Overall KPI Score,KPI Rank,Performance Category
0,MIT,USA,1,100.0,100,99,98,97,96,1,...,97,100.00,98.5,99,100.00,100.00,100.0,100.00,1,Excellent
1,University of Oxford,UK,2,99.2,99,98,97,96,95,2,...,96,97.96,98.0,98,97.96,97.56,96.0,97.17,2,Excellent
2,Stanford University,USA,3,98.4,99,98,97,96,95,4,...,96,95.92,97.5,98,95.92,95.12,96.0,95.68,3,Excellent
3,Harvard University,USA,4,97.6,98,97,96,95,94,6,...,95,93.88,97.0,97,93.88,92.68,92.0,92.85,4,Excellent
4,University of Cambridge,UK,5,96.8,98,97,96,95,94,3,...,95,91.84,97.0,97,91.84,92.68,92.0,92.17,5,Excellent
5,Caltech,USA,6,96.0,97,96,95,94,93,5,...,94,89.80,96.0,96,89.80,87.80,88.0,88.53,6,Excellent
6,Imperial College London,UK,7,95.2,97,96,95,94,93,7,...,94,87.76,96.0,96,87.76,87.80,88.0,87.85,7,Excellent
7,ETH Zurich,Switzerland,8,94.4,96,95,94,93,92,9,...,93,85.71,95.5,95,85.71,85.37,84.0,85.03,8,Excellent
8,National University of Singapore,Singapore,9,93.6,96,95,94,93,92,11,...,93,83.67,95.0,95,83.67,82.93,84.0,83.53,9,Excellent
9,UCL,UK,10,92.8,95,94,93,92,91,8,...,92,81.63,94.5,94,81.63,80.49,80.0,80.71,10,Excellent



File created successfully:
/content/Higher_Education_51_University_KPI.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

files.download("/content/Higher_Education_51_University_KPI.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>